In [2]:
# Importing required libraries

import pandas as pd
import numpy as np
import pickle
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [3]:
def load_txt(path):
    texts = []
    labels = []
    
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            
            text, label = line.split(";")
            texts.append(text)
            labels.append(label)
    
    return texts, labels

In [4]:
import os
print("Current working directory:")
print(os.getcwd())

print("\nFiles here:")
print(os.listdir())

Current working directory:
c:\Users\LENOVO\Desktop\Javin Programming\MindMirror\notebooks

Files here:
['train.ipynb']


In [5]:
train_texts, train_labels = load_txt("../kaggle_dataset/train.txt")
val_texts, val_labels = load_txt("../kaggle_dataset/val.txt")
test_texts, test_labels = load_txt("../kaggle_dataset/test.txt")

print("Train size:", len(train_texts))
print("Validation size:", len(val_texts))
print("Test size:", len(test_texts))

print("Unique labels:", set(train_labels))

Train size: 16000
Validation size: 2000
Test size: 2000
Unique labels: {'surprise', 'love', 'fear', 'sadness', 'anger', 'joy'}


In [6]:
# Basic normalization
train_texts = [text.lower().strip() for text in train_texts]
val_texts = [text.lower().strip() for text in val_texts]
test_texts = [text.lower().strip() for text in test_texts]

# Label encoding
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
train_labels_encoded = le.fit_transform(train_labels)
val_labels_encoded = le.transform(val_labels)
test_labels_encoded = le.transform(test_labels)

print("Classes:", le.classes_)

Classes: ['anger' 'fear' 'joy' 'love' 'sadness' 'surprise']


In [7]:
len(train_texts)

16000

In [8]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

vocab_size = 20000
max_len = 50

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(train_texts)

train_sequences = tokenizer.texts_to_sequences(train_texts)
val_sequences = tokenizer.texts_to_sequences(val_texts)
test_sequences = tokenizer.texts_to_sequences(test_texts)

X_train = pad_sequences(train_sequences, maxlen=max_len, padding='post')
X_val = pad_sequences(val_sequences, maxlen=max_len, padding='post')
X_test = pad_sequences(test_sequences, maxlen=max_len, padding='post')

print("X_train shape:", X_train.shape)

X_train shape: (16000, 50)


In [9]:
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=128),
    Bidirectional(LSTM(64, return_sequences=False)),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(6, activation='softmax')
])

model.compile(
    loss = 'sparse_categorical_crossentropy',
    optimizer = 'adam',
    metrics = ['accuracy']
)

model.build(input_shape=(None, max_len))

In [10]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 50, 128)        │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,667,462 (10.18 MB)

 Trainable params: 2,667,462 (10.18 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
history = model.fit(
    X_train,
    train_labels_encoded,
    validation_data=(X_val, val_labels_encoded),
    epochs=6,
    batch_size=32
)

Epoch 1/6
500/500 ━━━━━━━━━━━━━━━━━━━━ 12s 19ms/step - accuracy: 0.5226 - loss: 1.2057 - val_accuracy: 0.7645 - val_loss: 0.6959
Epoch 2/6
500/500 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.8534 - loss: 0.4425 - val_accuracy: 0.8960 - val_loss: 0.3299
Epoch 3/6
500/500 ━━━━━━━━━━━━━━━━━━━━ 10s 20ms/step - accuracy: 0.9304 - loss: 0.2120 - val_accuracy: 0.9005 - val_loss: 0.3647
Epoch 4/6
500/500 ━━━━━━━━━━━━━━━━━━━━ 9s 17ms/step - accuracy: 0.9562 - loss: 0.1448 - val_accuracy: 0.9000 - val_loss: 0.3675
Epoch 5/6
500/500 ━━━━━━━━━━━━━━━━━━━━ 9s 18ms/step - accuracy: 0.9685 - loss: 0.1026 - val_accuracy: 0.8930 - val_loss: 0.3870
Epoch 6/6
500/500 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.9729 - loss: 0.0840 - val_accuracy: 0.8900 - val_loss: 0.3829


In [12]:
test_loss, test_acc = model.evaluate(X_test, test_labels_encoded)
print("Test accuracy:", test_acc)
print("Test loss: ", test_loss)

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8905 - loss: 0.3866
Test accuracy: 0.890500009059906
Test loss:  0.38658562302589417


In [13]:
from sklearn.metrics import classification_report

y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

print(classification_report(test_labels_encoded, y_pred, target_names=le.classes_))

63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
              precision    recall  f1-score   support

       anger       0.91      0.89      0.90       275
        fear       0.87      0.83      0.85       224
         joy       0.92      0.90      0.91       695
        love       0.74      0.87      0.80       159
     sadness       0.97      0.91      0.94       581
    surprise       0.56      0.89      0.69        66

    accuracy                           0.89      2000
   macro avg       0.83      0.88      0.85      2000
weighted avg       0.90      0.89      0.89      2000



In [14]:
artifacts_path = Path("../artifacts")
artifacts_path.mkdir(exist_ok=True)

# Save model
model.save(artifacts_path / "model.keras")

# Save tokenizer
with open(artifacts_path / "tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

# Save label classes
with open(artifacts_path / "label_classes.pkl", "wb") as f:
    pickle.dump(le.classes_, f)

print("Artifacts saved successfully.")

Artifacts saved successfully.
